# Cleaned analysis notebook

This copy preserves the original analysis/calculation cells. Personal machine paths have been replaced with project-relative paths.
Place input workbooks and data in `data/`; generated files are written to `outputs/`. Create these folders before running.
Saved cell outputs were cleared so results are regenerated when the notebook is run.


## Pfirrmann Failure Analysis

1. Load file path

In [ ]:
import pandas as pd
import numpy as np

dice_file = "data/Original_dice_all_fold.xlsx"
pf_file = "data/radiological_gradings.csv"

dice_df = pd.read_excel(dice_file)
pf_df = pd.read_csv(pf_file)

print("Dice shape:", dice_df.shape)
print("Pfirrmann shape:", pf_df.shape)

print("\nDice columns:")
print(dice_df.columns.tolist())

print("\nPfirrmann columns:")
print(pf_df.columns.tolist())

2. Verify the complete Dice dataset

In [ ]:
import re

def extract_patient_id(filename):
    filename = str(filename)

    # Remove _0000.nii.gz or .nii
    filename = re.sub(r'\_0000.nii\.gz$', '', filename)
    filename = re.sub(r'\.nii\.gz$', '', filename)

    # If something like 83_0000 remains, take the part before _
    patient_id = filename.split('_')[0]

    return int(patient_id)

In [ ]:
dice_df["patient"] = dice_df["Case"].apply(extract_patient_id)

In [ ]:
print("Number of Dice rows:", len(dice_df))

# Check possible case identifier
for col in ["Case", "case", "Patient", "patient"]:
    if col in dice_df.columns:
        print(f"{col}: {dice_df[col].nunique()} unique values")

In [ ]:
#alternative verification
print("Dice Number range:")
print(
    dice_df["patient"].min(),
    "to",
    dice_df["patient"].max()
)

print(
    "Unique Dice Numbers:",
    dice_df["patient"].nunique()
)

print(
    dice_df[["Case", "patient"]].head(20)
)

3. Identify the Dice columns

In [ ]:
ivd_classes = [
    "Class_6",
    "Class_7",
    "Class_8",
    "Class_9",
    "Class_10",
    "Class_11"
]

missing = [c for c in ivd_classes if c not in dice_df.columns]

print("Missing IVD columns:", missing)

if missing:
    raise ValueError("Check your Dice column names.")

4.Standardize the patient/case identifier





In [ ]:
import pandas as pd
import numpy as np
import re

# --------------------------------------------------
# Create standardized patient ID in Dice dataframe
# --------------------------------------------------

def extract_patient_id(filename):
    filename = str(filename)

    # Remove .nii.gz
    filename = re.sub(r'\.nii\.gz$', '', filename)

    # Remove _0000 if present
    filename = re.sub(r'_0000$', '', filename)

    # Convert remaining number to integer
    return int(filename)


if "Case" in dice_df.columns:
    dice_df["patient"] = dice_df["Case"].apply(extract_patient_id)

elif "case" in dice_df.columns:
    dice_df["patient"] = dice_df["case"].apply(extract_patient_id)

else:
    raise ValueError("Could not find Case/case column in Dice dataframe.")


print("Dice patient identifiers:")
print(dice_df[["Case", "patient"]].head(20))

print("\nNumber of Dice rows:", len(dice_df))
print("Unique Dice patients:", dice_df["patient"].nunique())
print(
    "Patient ID range:",
    dice_df["patient"].min(),
    "to",
    dice_df["patient"].max()
)

In [ ]:
# --------------------------------------------------
# Standardize Pfirrmann columns
# --------------------------------------------------

pf_df.columns = (
    pf_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(pf_df.columns.tolist())

In [ ]:
# Make patient numeric
pf_df["patient"] = pd.to_numeric(
    pf_df["patient"],
    errors="coerce"
)

print("\nPfirrmann patient identifiers:")
print(pf_df["patient"].head(20))

print("\nUnique Pfirrmann patients:")
print(pf_df["patient"].nunique())

Patient matching check

In [ ]:
# --------------------------------------------------
# Check Dice–Pfirrmann patient matching
# --------------------------------------------------

dice_patients = set(dice_df["patient"].dropna().unique())
pf_patients = set(pf_df["patient"].dropna().unique())

common_patients = dice_patients.intersection(pf_patients)

print("Dice patients:", len(dice_patients))
print("Pfirrmann patients:", len(pf_patients))
print("Common patients:", len(common_patients))

print("\nDice patients NOT found in Pfirrmann:")
print(sorted(dice_patients - pf_patients))

print("\nPfirrmann patients NOT found in Dice:")
print(sorted(pf_patients - dice_patients))

IVD class mapping

In [ ]:
# --------------------------------------------------
# IVD Dice classes
# --------------------------------------------------

ivd_classes = [
    "Class_6",
    "Class_7",
    "Class_8",
    "Class_9",
    "Class_10",
    "Class_11"
]

missing = [
    c for c in ivd_classes
    if c not in dice_df.columns
]

print("Missing IVD columns:", missing)

if missing:
    raise ValueError("Check your Dice column names.")

In [ ]:
# --------------------------------------------------
# Confirm Dice class → anatomical disc mapping
# --------------------------------------------------

disc_map = {
    "Class_6": "L5_S1",
    "Class_7": "L4_L5",
    "Class_8": "L3_L4",
    "Class_9": "L2_L3",
    "Class_10": "L1_L2",
    "Class_11": "T12_L1"
}

print(
    pd.DataFrame(
        list(disc_map.items()),
        columns=["class_label", "disc_level"]
    )
)

 5. Convert the six IVD classes into long format


In [ ]:
# --------------------------------------------------
# Convert IVD Dice from wide → long format
# --------------------------------------------------

dice_ivd = dice_df[
    ["patient"] + ivd_classes
].copy()

dice_ivd[ivd_classes] = dice_ivd[ivd_classes].apply(
    pd.to_numeric,
    errors="coerce"
)

dice_long = dice_ivd.melt(
    id_vars="patient",
    value_vars=ivd_classes,
    var_name="class_label",
    value_name="dice"
)

dice_long["disc_level"] = (
    dice_long["class_label"].map(disc_map)
)

print("Long-format Dice shape:", dice_long.shape)

print(dice_long.head(12))

6. Map Pfirrmann IVD labels



In [ ]:
# --------------------------------------------------
# Pfirrmann IVD label → anatomical disc
# --------------------------------------------------

pf_df["ivd_label"] = pd.to_numeric(
    pf_df["ivd_label"],
    errors="coerce"
)

pf_df["disc_level"] = pf_df["ivd_label"].map({
    1: "T12_L1",
    2: "L1_L2",
    3: "L2_L3",
    4: "L3_L4",
    5: "L4_L5",
    6: "L5_S1"
})

pf_df["pfirrman_grade"] = pd.to_numeric(
    pf_df["pfirrman_grade"],
    errors="coerce"
)

print(
    pf_df[
        [
            "patient",
            "ivd_label",
            "disc_level",
            "pfirrman_grade"
        ]
    ].head(20)
)

 8. Merge Dice + Pfirrmann

In [ ]:
# --------------------------------------------------
# Merge Dice and Pfirrmann
# --------------------------------------------------

pf_small = pf_df[
    [
        "patient",
        "disc_level",
        "pfirrman_grade"
    ]
].copy()

merged = dice_long.merge(
    pf_small,
    on=["patient", "disc_level"],
    how="left"
)

print("Merged shape:", merged.shape)

print(
    merged.head(20)
)

 9. Validate the merge

In [ ]:
print("\nTotal Dice observations:")
print(len(merged))

print("\nPfirrmann available:")
print(merged["pfirrman_grade"].notna().sum())

print("\nPfirrmann missing:")
print(merged["pfirrman_grade"].isna().sum())

print("\nPfirrmann grade distribution:")
print(
    merged["pfirrman_grade"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nNumber of Dice patients:")
print(merged["patient"].nunique())

print("\nNumber of patients per Pfirrmann grade:")
print(
    merged.groupby("pfirrman_grade")["patient"]
    .nunique()
)

10. Check for duplicate patient-disc combinations

In [ ]:
duplicates = merged[
    merged.duplicated(
        subset=["patient", "disc_level"],
        keep=False
    )
]

print(
    "Duplicate patient-disc rows:",
    len(duplicates)
)

if len(duplicates) > 0:
    print(duplicates)

11. Define segmentation failure

In [ ]:
merged["failure"] = (
    merged["dice"] < 0.70
)

print(
    "Total IVD observations:",
    len(merged)
)

print(
    "Total IVD failures:",
    merged["failure"].sum()
)

print(
    "Overall IVD failure rate:",
    merged["failure"].mean() * 100,
    "%"
)

12. Failure rate by Pfirrmann grade

In [ ]:
failure_summary = (
    merged
    .dropna(subset=["pfirrman_grade"])
    .groupby("pfirrman_grade")
    .agg(
        total_discs=("failure", "size"),
        failed_discs=("failure", "sum")
    )
    .reset_index()
)

failure_summary["failure_rate_percent"] = (
    failure_summary["failed_discs"]
    / failure_summary["total_discs"]
    * 100
)

print(failure_summary)

13. Calculate Dice distribution by Pfirrmann grade

In [ ]:
  dice_summary = (
    merged
    .dropna(subset=["pfirrman_grade"])
    .groupby("pfirrman_grade")["dice"]
    .agg(
        N="count",
        Mean="mean",
        SD="std",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

dice_summary["IQR"] = (
    dice_summary["Q3"]
    - dice_summary["Q1"]
)

print(dice_summary.round(4))

15. Dice distribution by Pfirrmann grade (figure)

In [ ]:
import matplotlib.pyplot as plt

plot_data = []

grades = [1, 2, 3, 4, 5]

for grade in grades:
    values = merged.loc[
        merged["pfirrman_grade"] == grade,
        "dice"
    ].dropna().values

    plot_data.append(values)

fig, ax = plt.subplots(figsize=(8, 6))

ax.boxplot(
    plot_data,
    positions=grades,
    widths=0.55,
    showfliers=False
)

# Add individual observations with jitter
rng = np.random.default_rng(42)

for grade, values in zip(grades, plot_data):

    jitter = rng.uniform(
        -0.10,
        0.10,
        size=len(values)
    )

    ax.scatter(
        grade + jitter,
        values,
        s=15,
        alpha=0.35
    )

ax.set_xticks(grades)

ax.set_xticklabels([
    "I",
    "II",
    "III",
    "IV",
    "V"
])

ax.set_xlabel("Pfirrmann grade")
ax.set_ylabel("IVD Dice coefficient")

ax.set_title(
    "IVD segmentation performance by Pfirrmann grade"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    "outputs/dice_by_pfirrmann.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "outputs/dice_by_pfirrmann.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

16. Failure rate by Pfirrmann grade

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

grades_present = failure_summary["pfirrman_grade"].astype(int)

ax.bar(
    range(len(failure_summary)),
    failure_summary["failure_rate_percent"]
)

ax.set_xticks(range(len(failure_summary)))

ax.set_xticklabels(
    grades_present.astype(str)
)

ax.set_xlabel("Pfirrmann grade")
ax.set_ylabel("Segmentation failure rate (%)")

ax.set_title(
    "IVD segmentation failure rate by Pfirrmann grade"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Add percentage labels
for i, value in enumerate(
    failure_summary["failure_rate_percent"]
):
    ax.text(
        i,
        value + 0.8,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()

plt.savefig(
    "outputs/failure_rate_by_pfirrmann.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "outputs/failure_rate_by_pfirrmann.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

17. Statistical analysis
*   Mixed-effects model for Dice








In [ ]:
import statsmodels.formula.api as smf

In [ ]:
model_df = merged.dropna(
    subset=[
        "dice",
        "pfirrman_grade",
        "patient"
    ]
).copy()

model_df["pfirrman_grade"] = (
    model_df["pfirrman_grade"].astype(int)
)

model = smf.mixedlm(
    "dice ~ C(pfirrman_grade)",
    data=model_df,
    groups=model_df["patient"]
)

result = model.fit(
    reml=False
)

print(result.summary())

GEE failure model


In [ ]:
import statsmodels.api as sm

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

gee_df = merged.dropna(
    subset=[
        "failure",
        "pfirrman_grade",
        "patient"
    ]
).copy()

gee_df["pfirrman_grade"] = (
    gee_df["pfirrman_grade"].astype(int)
)

gee_model = smf.gee(
    "failure ~ C(pfirrman_grade)",
    groups="patient",
    data=gee_df,
    family=sm.families.Binomial()
)

gee_result = gee_model.fit()

print(gee_result.summary())

18. Save

In [ ]:
# Complete merged disc-level dataset
merged.to_csv(
    "outputs/pfirrmann_complete_disc_level_dataset.csv",
    index=False
)

# Dice summary
dice_summary.to_csv(
    "outputs/pfirrmann_dice_summary.csv",
    index=False
)

# Failure summary
failure_summary.to_csv(
    "outputs/pfirrmann_failure_summary.csv",
    index=False
)

print("All analysis files saved.")

cluster-aware mixed-effects model (primary omnibus model)

In [ ]:
model = smf.mixedlm(
    "dice ~ C(pfirrman_grade)",
    data=model_df,
    groups=model_df["patient"]
)

result = model.fit(reml=False)

print(result.summary())

Omnibus test

In [ ]:
import scipy.stats as stats

# Full model
full_model = smf.mixedlm(
    "dice ~ C(pfirrman_grade)",
    data=model_df,
    groups=model_df["patient"]
)

full_result = full_model.fit(reml=False);

# Reduced model without Pfirrmann grade
null_model = smf.mixedlm(
    "dice ~ 1",
    data=model_df,
    groups=model_df["patient"]
)

null_result = null_model.fit(reml=False);

# Likelihood-ratio test
lr_stat = 2 * (full_result.llf - null_result.llf)

df_diff = full_result.df_modelwc - null_result.df_modelwc

p_value = stats.chi2.sf(lr_stat, df_diff)

print("Likelihood-ratio omnibus test")
print("--------------------------------")
print(f"LR statistic = {lr_stat:.4f}")
print(f"df = {df_diff}")
print(f"p-value = {p_value:.6g}")

Estimated marginal means for each grade

In [ ]:
grades = [1, 2, 3, 4, 5]

emm = []

for grade in grades:
    new_data = pd.DataFrame({
        "pfirrman_grade": [grade]
    })

    # Fixed-effect prediction only
    pred = full_result.predict(new_data)[0]

    emm.append({
        "Grade": grade,
        "Estimated_Mean_Dice": pred
    })

emm_df = pd.DataFrame(emm)

print(emm_df)

Pairwise comparisons with multiplicity adjustment

In [ ]:
from itertools import combinations
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# Fixed-effect coefficient vector
beta = full_result.fe_params.values

# Covariance matrix of fixed effects
cov_beta = full_result.cov_params().loc[
    full_result.fe_params.index,
    full_result.fe_params.index
].values

# Create design rows for each Pfirrmann grade
design = {}

for grade in grades:
    if grade == 1:
        # Reference grade
        row = np.array([1, 0, 0, 0, 0])
    elif grade == 2:
        row = np.array([1, 1, 0, 0, 0])
    elif grade == 3:
        row = np.array([1, 0, 1, 0, 0])
    elif grade == 4:
        row = np.array([1, 0, 0, 1, 0])
    elif grade == 5:
        row = np.array([1, 0, 0, 0, 1])

    design[grade] = row

pairwise_results = []

for g1, g2 in combinations(grades, 2):

    contrast = design[g2] - design[g1]

    estimate = np.dot(contrast, beta)

    variance = np.dot(
        contrast,
        np.dot(cov_beta, contrast)
    )

    se = np.sqrt(variance)

    z_value = estimate / se

    p_raw = 2 * norm.sf(abs(z_value))

    ci_low = estimate - 1.96 * se
    ci_high = estimate + 1.96 * se

    pairwise_results.append({
        "Comparison": f"Grade {g2} vs Grade {g1}",
        "Difference": estimate,
        "SE": se,
        "z": z_value,
        "p_raw": p_raw,
        "CI_low": ci_low,
        "CI_high": ci_high
    })

pairwise_df = pd.DataFrame(pairwise_results)

# Tukey/Holm-style multiplicity adjustment
pairwise_df["p_adjusted"] = multipletests(
    pairwise_df["p_raw"],
    method="holm"
)[1]

print(pairwise_df.round(4))

In [ ]:
print(full_result.fe_params.index)
print(full_result.cov_params().index)

In [ ]:
  final_pairwise = pairwise_df.copy()

final_pairwise["Difference"] = final_pairwise["Difference"].round(4)
final_pairwise["SE"] = final_pairwise["SE"].round(4)
final_pairwise["CI_low"] = final_pairwise["CI_low"].round(4)
final_pairwise["CI_high"] = final_pairwise["CI_high"].round(4)
final_pairwise["p_raw"] = final_pairwise["p_raw"].round(4)
final_pairwise["p_adjusted"] = final_pairwise["p_adjusted"].round(4)

final_pairwise

In [ ]:
print("OMNIBUS")
print(f"LR = {lr_stat:.4f}, df = {df_diff}, p = {p_value:.6g}")

print("\nESTIMATED MARGINAL MEANS")
print(emm_df)

print("\nPAIRWISE")
print(pairwise_df)